# Why Group-Based Splitting Matters
## A Methodological Validation Study

This notebook demonstrates why group-based data splitting is essential
for honest evaluation of document fraud detection models.

### The problem with random splitting
In document image datasets, multiple images come from the same physical
document (different scans, angles, lighting conditions). If images from
the same document appear in both training and test sets, the model
effectively memorises document identity rather than learning fraud patterns.
This inflates performance metrics — the model has already "seen" the
document during training.

### What this notebook shows
Two experiments are compared on identical models and features:

| Experiment | Split strategy | CV strategy | Expected result |
|---|---|---|---|
| Old (leaky) | Random row split | StratifiedKFold | Inflated metrics |
| New (clean) | Group-based split | StratifiedGroupKFold | Honest metrics |

### Thesis relevance
This analysis justifies the methodological decision to use group-based
splits throughout the thesis. The difference in performance between the
two approaches quantifies the leakage effect and shows how much reported
accuracy would have been artificially inflated without this correction.

## 0 · Imports & setup

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, r"C:\Users\Admin\Desktop\thesis\multimodal-fraud-detection-thesis\src")

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

PROJECT_ROOT = Path().resolve().parent
DATA_DIR     = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR  = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
NB_RESULTS   = PROJECT_ROOT / "notebook" / "results" / "split_comparison"
NB_RESULTS.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 1 · The problem: document groups in image datasets

Before running any model, let's look at the data structure to understand
why random splitting is problematic.

In [ ]:
# Load the full image dataset
img_df = pd.read_csv(DATA_DIR / "image_dataset_final.csv")

print(f"Total images: {len(img_df):,}")
print(f"Unique groups: {img_df['group_key'].nunique():,}")
print(f"\nImages per group (statistics):")
imgs_per_group = img_df.groupby("group_key").size()
print(imgs_per_group.describe().round(1))

print(f"\nSource dataset breakdown:")
display(pd.crosstab(img_df["source_dataset"], img_df["image_class"]))

In [ ]:
# Show that one group_key = one physical document with multiple images
example_group = img_df[img_df["source_dataset"]=="fantasyid"]["group_key"].iloc[0]
group_images = img_df[img_df["group_key"]==example_group]

print(f"Example group_key: {example_group}")
print(f"Images in this group: {len(group_images)}")
print(group_images[["image_path","image_class","source_type"]].to_string(index=False))

print(f"\nIf this group is split across train/test:")
print("  → model sees this document in training")
print("  → model is tested on the SAME document")
print("  → model memorises document identity, not fraud patterns")

## 2 · Old splits: measuring the leakage

The old splits (`image_train.csv`, `image_test.csv`) were created by
random row sampling without respecting document groups.

In [ ]:
old_train_path = DATA_DIR / "image_train.csv"
old_test_path  = DATA_DIR / "image_test.csv"
new_train_path = DATA_DIR / "image_train_group_test.csv"
new_test_path  = DATA_DIR / "image_test_group_test.csv"

# Check which files exist
for path in [old_train_path, old_test_path, new_train_path, new_test_path]:
    print(f"  {path.name}: {'✅ exists' if path.exists() else '❌ missing'}")

In [ ]:
if old_train_path.exists() and old_test_path.exists():
    old_train = pd.read_csv(old_train_path)
    old_test  = pd.read_csv(old_test_path)

    # Group overlap
    old_overlap_groups = set(old_train["group_key"]) & set(old_test["group_key"])
    old_overlap_images = set(old_train["image_path"]) & set(old_test["image_path"])

    print("=== OLD SPLITS (random row split) ===")
    print(f"Train rows: {len(old_train):,}  |  Test rows: {len(old_test):,}")
    print(f"Train groups: {old_train['group_key'].nunique():,}  |  Test groups: {old_test['group_key'].nunique():,}")
    print(f"")
    print(f"Group overlap (same document in train AND test): {len(old_overlap_groups):,} groups")
    print(f"Image overlap (exact same image in train AND test): {len(old_overlap_images):,} images")

    if len(old_overlap_groups) > 0:
        overlap_pct = len(old_overlap_groups) / old_test["group_key"].nunique() * 100
        print(f"  → {overlap_pct:.1f}% of test groups were seen during training")
        print(f"  ⚠️  This means performance metrics are INFLATED")
else:
    print("Old split files not found — showing new splits only")

## 3 · New splits: group-based splitting

The new splits (`image_train_group_test.csv`, etc.) ensure that no
document group appears in more than one split.

In [ ]:
new_train = pd.read_csv(new_train_path)
new_test  = pd.read_csv(new_test_path)

new_overlap_groups = set(new_train["group_key"]) & set(new_test["group_key"])
new_overlap_images = set(new_train["image_path"]) & set(new_test["image_path"])

print("=== NEW SPLITS (group-based split) ===")
print(f"Train rows: {len(new_train):,}  |  Test rows: {len(new_test):,}")
print(f"Train groups: {new_train['group_key'].nunique():,}  |  Test groups: {new_test['group_key'].nunique():,}")
print(f"")
print(f"Group overlap: {len(new_overlap_groups):,}  {'✅ clean' if len(new_overlap_groups)==0 else '⚠️ LEAKAGE'}")
print(f"Image overlap: {len(new_overlap_images):,}  {'✅ clean' if len(new_overlap_images)==0 else '⚠️ LEAKAGE'}")

if len(new_overlap_groups) == 0:
    print(f"  → No document appears in both train and test")
    print(f"  → Model must generalise to UNSEEN documents")

## 4 · Visualising the difference

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: overlap comparison
labels  = ["Old splits
(random)", "New splits
(group-based)"]
overlaps = [len(old_overlap_groups) if old_train_path.exists() else 0,
            len(new_overlap_groups)]
colours = ["#C44E52" if o > 0 else "#55A868" for o in overlaps]

bars = axes[0].bar(labels, overlaps, color=colours, edgecolor="white", width=0.4)
for bar, val in zip(bars, overlaps):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height() + max(overlaps)*0.02,
                 str(val), ha="center", va="bottom", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of overlapping groups")
axes[0].set_title("Document group overlap\nbetween train and test")
axes[0].spines[["top","right"]].set_visible(False)

# Right: group distribution
new_train_src = new_train.groupby("source_dataset")["group_key"].nunique()
new_test_src  = new_test.groupby("source_dataset")["group_key"].nunique()

src_df = pd.DataFrame({"train": new_train_src, "test": new_test_src}).fillna(0)
x = np.arange(len(src_df)); width = 0.35
axes[1].bar(x-width/2, src_df["train"], width, label="Train",
            color="#4C72B0", alpha=0.85)
axes[1].bar(x+width/2, src_df["test"],  width, label="Test",
            color="#DD8452", alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(src_df.index)
axes[1].set_ylabel("Number of unique document groups")
axes[1].set_title("Group distribution by source\n(new group-based splits)")
axes[1].legend(); axes[1].spines[["top","right"]].set_visible(False)

fig.suptitle("Group-Based Splitting: Eliminating Data Leakage", fontsize=13)
fig.tight_layout()
fig.savefig(NB_RESULTS / "split_comparison_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 5 · Model comparison: old vs new splits

Now we train the same model (ResNet-18 embeddings + Logistic Regression)
on both split strategies and compare the results.

If the old splits gave inflated metrics, we expect:
- Old splits → higher F1 / ROC-AUC (leaked document identity)
- New splits → lower but honest F1 / ROC-AUC (true generalisation)

In [ ]:
results_path = RESULTS_DIR / "visual_pipeline_compare_results.json"

if results_path.exists():
    print("Loading existing results from:", results_path)
    with open(results_path) as f:
        results = json.load(f)
    print("Results loaded successfully.")
    print("Experiments found:", list(results.keys()))
else:
    print("Results file not found. Running the comparison script...")
    print("This may take several minutes (extracting CNN embeddings).")
    import subprocess
    result = subprocess.run(
        ["python", str(PROJECT_ROOT / "src" / "visual_pipeline_compare.py")],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT)
    )
    print(result.stdout[-3000:] if result.stdout else "(no output)")
    if result.returncode != 0:
        print("ERRORS:"); print(result.stderr[-1000:])
    else:
        with open(results_path) as f:
            results = json.load(f)
        print("Done.")

## 6 · Results: the inflation effect

In [ ]:
if "results" not in dir() or not results:
    print("No results available — run cell 5 first.")
else:
    # Build comparison dataframe
    rows = []
    for exp_name, r in results.items():
        label = ("Old splits\n(random CV)" if "old" in exp_name
                 else "New splits\n(group CV)")
        m = r["test_metrics"]
        rows.append({
            "experiment":  label,
            "split_type":  "Old (leaky)" if "old" in exp_name else "New (clean)",
            "train_rows":  r["train_rows"],
            "test_rows":   r["test_rows"],
            "overlap":     r["train_test_overlap"],
            "best_cv_f1":  r["best_cv_score_f1"],
            "accuracy":    m["accuracy"],
            "precision":   m["precision"],
            "recall":      m["recall"],
            "f1":          m["f1"],
            "roc_auc":     m["roc_auc"],
        })

    cmp_df = pd.DataFrame(rows)
    display(cmp_df[["experiment","split_type","overlap",
                     "accuracy","precision","recall","f1","roc_auc"]].round(4))
    cmp_df.to_csv(NB_RESULTS / "split_comparison_results.csv", index=False)

    # Calculate inflation
    if len(cmp_df) == 2:
        old_f1  = cmp_df[cmp_df["split_type"]=="Old (leaky)"]["f1"].values[0]
        new_f1  = cmp_df[cmp_df["split_type"]=="New (clean)"]["f1"].values[0]
        old_auc = cmp_df[cmp_df["split_type"]=="Old (leaky)"]["roc_auc"].values[0]
        new_auc = cmp_df[cmp_df["split_type"]=="New (clean)"]["roc_auc"].values[0]

        print(f"\n=== INFLATION EFFECT ===")
        print(f"F1 inflation:      {old_f1:.4f} → {new_f1:.4f}  "
              f"(overestimated by {old_f1-new_f1:+.4f})")
        print(f"ROC-AUC inflation: {old_auc:.4f} → {new_auc:.4f}  "
              f"(overestimated by {old_auc-new_auc:+.4f})")
        print(f"\nConclusion: Random splitting OVERESTIMATES performance by "
              f"{(old_f1-new_f1)*100:.1f} F1 points.")

## 7 · Performance comparison chart

In [ ]:
if "cmp_df" in dir() and len(cmp_df) > 0:
    metrics = ["accuracy","precision","recall","f1","roc_auc"]
    x = np.arange(len(metrics)); width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    colours_exp = {"Old (leaky)":"#C44E52","New (clean)":"#55A868"}
    for i, (_, row) in enumerate(cmp_df.iterrows()):
        offset = -width/2 if i == 0 else width/2
        bars = axes[0].bar(x + offset,
                           [row[m] for m in metrics],
                           width, label=row["split_type"],
                           color=colours_exp[row["split_type"]],
                           alpha=0.85)

    axes[0].set_xticks(x)
    axes[0].set_xticklabels(["Accuracy","Precision","Recall","F1","ROC-AUC"])
    axes[0].set_ylim(0.5, 1.1); axes[0].set_ylabel("Score")
    axes[0].set_title("Old (leaky) vs New (clean) splits\nsame model, same features")
    axes[0].legend(); axes[0].spines[["top","right"]].set_visible(False)

    # Inflation bars
    if len(cmp_df) == 2:
        old_row = cmp_df[cmp_df["split_type"]=="Old (leaky)"].iloc[0]
        new_row = cmp_df[cmp_df["split_type"]=="New (clean)"].iloc[0]
        inflation = [old_row[m] - new_row[m] for m in metrics]
        bar_c = ["#C44E52" if v > 0.005 else "#55A868" for v in inflation]
        bars2 = axes[1].bar(metrics, inflation, color=bar_c, edgecolor="white")
        for bar, val in zip(bars2, inflation):
            axes[1].text(bar.get_x()+bar.get_width()/2,
                         bar.get_height() + 0.001,
                         f"{val:+.3f}", ha="center", va="bottom", fontsize=9)
        axes[1].axhline(0, color="black", lw=0.8)
        axes[1].set_xticklabels(["Acc","Prec","Recall","F1","ROC-AUC"])
        axes[1].set_ylabel("Old − New  (positive = inflation)")
        axes[1].set_title("Performance inflation\nfrom data leakage")
        axes[1].spines[["top","right"]].set_visible(False)

    fig.suptitle("Impact of Group-Based Splitting on Reported Performance",
                 fontsize=13)
    fig.tight_layout()
    fig.savefig(NB_RESULTS / "performance_inflation.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8 · Why the CV strategy also matters

Beyond the train/test split, the cross-validation strategy during hyperparameter
tuning also matters. Using `StratifiedKFold` on grouped data during CV creates
the same leakage problem within the training set.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")

table_data = [
    ["Strategy", "Split type", "CV type", "Leakage risk", "Use case"],
    ["Old baseline", "Random rows", "StratifiedKFold", "HIGH ⚠️",
     "Inflated benchmark"],
    ["New (this thesis)", "Group-based", "StratifiedGroupKFold", "NONE ✅",
     "Honest evaluation"],
]

col_labels = table_data[0]
cell_data  = table_data[1:]

tbl = ax.table(
    cellText=cell_data,
    colLabels=col_labels,
    cellLoc="center", loc="center",
)
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 2.2)

# Colour the leakage column
for j, label in enumerate(col_labels):
    tbl[0, j].set_facecolor("#2c3e50")
    tbl[0, j].set_text_props(color="white", fontweight="bold")
for i in range(1, len(cell_data)+1):
    if "HIGH" in cell_data[i-1][3]:
        tbl[i, 3].set_facecolor("#f8d7da")
    elif "NONE" in cell_data[i-1][3]:
        tbl[i, 3].set_facecolor("#d4edda")

ax.set_title("Splitting and Cross-Validation Strategy Comparison",
             fontsize=13, pad=25)
fig.tight_layout()
fig.savefig(NB_RESULTS / "strategy_table.png", dpi=150, bbox_inches="tight")
plt.show()

## 9 · Discussion: what this means for the thesis

### Key finding
The comparison between random and group-based splitting shows that random
splitting artificially inflates model performance. This happens because:

1. **Same document in train and test**: the model memorises visual features
   specific to individual documents rather than learning general fraud patterns.

2. **CV leakage compounds the effect**: when `StratifiedKFold` is used during
   hyperparameter tuning, the selected hyperparameters are optimised for a
   leaky evaluation, further inflating reported scores.

### Methodological implication
All experiments in this thesis use group-based splitting with
`StratifiedGroupKFold` for cross-validation. This ensures that:
- Model performance reflects true generalisation to **unseen documents**
- Reported metrics are not inflated by document identity memorisation
- The fraud detection capability is evaluated fairly

### How to write this in your thesis
> *Prior to model evaluation, the impact of data splitting strategy was
> investigated. Document images in the dataset are organised into groups
> corresponding to individual physical documents, where multiple images
> may exist per document. A comparison between random row-level splitting
> and group-based splitting revealed that random splitting produced
> inflated performance estimates, with F1 overestimated by X points and
> ROC-AUC by Y points. This leakage occurs because images from the same
> document appear in both training and test sets, allowing the model to
> memorise document identity rather than learning generalised fraud patterns.
> Consequently, all experiments in this thesis employ group-based splitting
> to ensure honest evaluation of model generalisation.*

In [ ]:
# Save a summary for the thesis
if "cmp_df" in dir() and len(cmp_df) == 2:
    old_row = cmp_df[cmp_df["split_type"]=="Old (leaky)"].iloc[0]
    new_row = cmp_df[cmp_df["split_type"]=="New (clean)"].iloc[0]

    summary = {
        "old_split": {
            "f1":      round(float(old_row["f1"]), 4),
            "roc_auc": round(float(old_row["roc_auc"]), 4),
            "overlap": int(old_row["overlap"]),
        },
        "new_split": {
            "f1":      round(float(new_row["f1"]), 4),
            "roc_auc": round(float(new_row["roc_auc"]), 4),
            "overlap": int(new_row["overlap"]),
        },
        "inflation": {
            "f1_points":      round(float(old_row["f1"] - new_row["f1"]), 4),
            "roc_auc_points": round(float(old_row["roc_auc"] - new_row["roc_auc"]), 4),
        }
    }
    with open(NB_RESULTS / "inflation_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Thesis numbers:")
    print(f"  Random split F1:      {summary['old_split']['f1']}")
    print(f"  Group split F1:       {summary['new_split']['f1']}")
    print(f"  F1 inflation:         {summary['inflation']['f1_points']:+.4f}")
    print(f"  ROC-AUC inflation:    {summary['inflation']['roc_auc_points']:+.4f}")
    print(f"\nAll figures saved to: {NB_RESULTS}")